# Origami RL — GRPO Training

Train an LLM to generate FOLD crease patterns using OpenEnv + Unsloth + TRL.

Follows the [2048 OpenEnv notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/OpenEnv_gpt_oss_(20B)_Reinforcement_Learning_2048_Game.ipynb) pattern exactly:
1. Connect to origami environment server via OpenEnv client
2. LLM generates FOLD JSON crease patterns
3. Reward functions call the server — physics simulation + shape matching
4. GRPO updates policy based on relative rewards

## 1. Install Dependencies

In [ ]:
import os, subprocess, importlib.util

# Install uv (fast pip replacement)
!pip install --upgrade -qqq uv

# Install unsloth + dependencies
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    # Fresh env (Colab) — install everything
    try: import numpy; get_numpy = f"numpy=={numpy.__version__}"
    except: get_numpy = "numpy"
    !uv pip install \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} torchvision bitsandbytes "transformers==4.56.2" trackio \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    # Has torch but no unsloth (RunPod) — install unsloth
    !pip install unsloth trackio
!pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# Verify
import unsloth
print(f"unsloth {unsloth.__version__} installed OK")

## 2. Clone Origami Env + Setup Paths

In [ ]:
%%capture
!pip install -qqq fastapi uvicorn requests websockets
!pip install -qqq "openenv-core[core]>=0.2.1"
!git clone https://github.com/Prasanna721/Origami.git > /dev/null 2>&1

import subprocess, sys, os
from pathlib import Path

# Add origami_env to Python path
working_directory = str(Path.cwd().absolute() / "Origami" / "origami_env")
sys.path.insert(0, working_directory)
print(f"Working directory: {working_directory}")

In [ ]:
from client import OrigamiEnv
from origami_server.models import OrigamiAction, OrigamiObservation, OrigamiState
print("Origami OpenEnv modules loaded.")

## 3. Load Model + QLoRA

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 768  # FOLD JSON is compact
lora_rank = 4

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen3-14B",
    load_in_4bit = True,
    max_seq_length = max_seq_length,
    offload_embedding = True,
)

## 4. LoRA Adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 5. Connect to OpenEnv Server

The origami environment is deployed on Railway. We connect via OpenEnv WebSocket client.

In [ ]:
SERVER_URL = "https://origami-env-production.up.railway.app"

# Global client — reused across reward function calls
global openenv_client
openenv_client = None

def get_env():
    """Get or create OpenEnv client. Reconnects if connection dropped."""
    global openenv_client
    if openenv_client is not None:
        try:
            # Test if connection is alive
            openenv_client._ensure_connected()
            return openenv_client
        except Exception:
            try: openenv_client.close()
            except: pass
            openenv_client = None
    openenv_client = OrigamiEnv(base_url=SERVER_URL)
    return openenv_client

In [ ]:
# Test the connection
import requests
health = requests.get(f"{SERVER_URL}/health").json()
print(f"Server: {health}")

env = get_env()
result = env.reset(task_name="triangle")
print(f"Reset OK: done={result.done}, reward={result.reward}")
print(f"Task: {result.observation.task}")

# Test step with known-good fold
result = env.step(OrigamiAction(fold_data={
    "vertices_coords": [[0,0],[1,0],[1,1],[0,1]],
    "edges_vertices": [[0,1],[1,2],[2,3],[3,0],[0,2]],
    "edges_assignment": ["B","B","B","B","V"],
    "edges_foldAngle": [0,0,0,0,180]
}))
print(f"Step OK: reward={result.reward}, similarity={result.observation.shape_similarity}")

## 6. Prompt + Dataset

In [ ]:
# Fetch ALL tasks from deployed server
all_tasks = requests.get(f"{SERVER_URL}/tasks").json()
print(f"Tasks available: {list(all_tasks.keys())}\n")

PROMPT_TEMPLATE = """You are an origami designer. Generate a FOLD-format crease pattern
that, when folded, produces the target shape described below.

Target: {description}
Paper size: {width} x {height}

Output a JSON object with these exact fields:
- vertices_coords: [[x, y], ...] — 2D positions on the flat paper
- edges_vertices: [[v1, v2], ...] — pairs of vertex indices forming edges
- edges_assignment: ["B"|"M"|"V", ...] — B=boundary, M=mountain fold, V=valley fold
- edges_foldAngle: [angle, ...] — fold angles in degrees (V: 180, M: -180, B: 0)

Rules:
- Boundary edges (B) must outline the paper rectangle
- At least one fold crease (M or V) must exist
- All vertex indices must be valid (0 to N-1)

Output ONLY the JSON object wrapped in ```json ... ``` markers."""

# Build one prompt per task
task_prompts = {}
for name, info in all_tasks.items():
    task_prompts[name] = PROMPT_TEMPLATE.format(
        description=info["description"],
        width=info["paper"]["width"],
        height=info["paper"]["height"],
    ).strip()
    print(f"  {name} (difficulty {info['difficulty']}): {info['description']}")

# Show one example
print(f"\n--- Example prompt (triangle) ---\n{task_prompts['triangle'][:200]}...")

In [ ]:
# Test generation before training (pick any task)
prompt = task_prompts["triangle"]
text = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    tokenize = False,
    add_generation_prompt = True,
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors="pt").to("cuda"),
    temperature = 1.0,
    max_new_tokens = 256,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

In [ ]:
from datasets import Dataset

# Mix all 4 tasks — 250 copies each = 1000 total
rows = []
for name, prompt_text in task_prompts.items():
    for _ in range(250):
        rows.append({
            "prompt": [{"role": "user", "content": prompt_text}],
            "task_name": name,
        })

# Shuffle so tasks are interleaved
import random
random.seed(3407)
random.shuffle(rows)

dataset = Dataset.from_list(rows)

# Compute max prompt length across all tasks
maximum_length = max(
    len(tokenizer.apply_chat_template(
        [{"role": "user", "content": p}], add_generation_prompt=True
    ))
    for p in task_prompts.values()
)
print(f"Max prompt token length: {maximum_length}")
print(f"Dataset: {len(dataset)} rows ({len(task_prompts)} tasks × 250)")
print(f"Task distribution: {dict(zip(*np.unique(dataset['task_name'], return_counts=True)))}")

## 7. Reward Functions

Two reward functions (same pattern as 2048 notebook):
- `valid_fold` — local JSON structure check (fast, no server call)
- `shape_match` — calls the origami server via OpenEnv, submits the fold, returns similarity × 20

In [ ]:
import json, re

def extract_fold_json(response):
    """Extract FOLD JSON from LLM response text."""
    # Try fenced code block
    match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", response, re.DOTALL)
    if match:
        try: return json.loads(match.group(1))
        except json.JSONDecodeError: pass
    # Try raw JSON with vertices_coords
    match = re.search(r"\{[^{}]*\"vertices_coords\"[^{}]*\}", response, re.DOTALL)
    if match:
        try: return json.loads(match.group(0))
        except json.JSONDecodeError: pass
    # Try whole response
    try:
        data = json.loads(response.strip())
        if isinstance(data, dict) and "vertices_coords" in data:
            return data
    except (json.JSONDecodeError, ValueError): pass
    return None

print(extract_fold_json('```json\n{"vertices_coords": [[0,0]]}\n```'))

In [ ]:
def valid_fold(completions, **kwargs):
    """Reward 1: Does the LLM output parse as valid FOLD JSON?
    +1.0 valid, -0.5 parseable but invalid structure, -2.0 unparseable."""
    scores = []
    for completion in completions:
        response = completion[0]["content"]
        fold_data = extract_fold_json(response)
        if fold_data is None:
            scores.append(-2.0); continue
        required = {"vertices_coords", "edges_vertices", "edges_assignment"}
        if not required.issubset(fold_data.keys()):
            scores.append(-0.5); continue
        verts = fold_data.get("vertices_coords", [])
        edges = fold_data.get("edges_vertices", [])
        assigns = fold_data.get("edges_assignment", [])
        if len(edges) != len(assigns):
            scores.append(-0.5); continue
        if not any(a in ("M", "V") for a in assigns) or not any(a == "B" for a in assigns):
            scores.append(-0.5); continue
        n = len(verts)
        if not all(0 <= e[0] < n and 0 <= e[1] < n and e[0] != e[1] for e in edges):
            scores.append(-0.5); continue
        scores.append(1.0)
    return scores

In [ ]:
import numpy as np
global PRINTER
PRINTER = 0

# Build reverse lookup: description → task_name
_desc_to_task = {info["description"]: name for name, info in all_tasks.items()}

def _get_task_from_prompt(prompts, idx):
    """Extract task name from the prompt text."""
    try:
        prompt_text = prompts[idx][0]["content"] if prompts else ""
        for desc, name in _desc_to_task.items():
            if desc in prompt_text:
                return name
    except (IndexError, KeyError, TypeError):
        pass
    return "triangle"  # fallback

def shape_match(completions, prompts=None, **kwargs):
    """Reward 2: Submit fold to origami server, get shape similarity reward.
    Calls the deployed OpenEnv server via WebSocket client.
    Determines which task from the prompt text."""
    global PRINTER, openenv_client
    scores = []
    for i, completion in enumerate(completions):
        response = completion[0]["content"]
        fold_data = extract_fold_json(response)
        task_name = _get_task_from_prompt(prompts, i)

        if PRINTER % 5 == 0:
            print(f"[{task_name}] {json.dumps(fold_data, indent=2)[:200] if fold_data else 'UNPARSEABLE'}")
        PRINTER += 1

        if fold_data is None:
            scores.append(0.0)
            continue
        try:
            env = get_env()
            env.reset(task_name=task_name)
            result = env.step(OrigamiAction(fold_data=fold_data))
            reward = result.reward if result.reward is not None else 0.0
            print(f"[{task_name}] similarity={result.observation.shape_similarity:.3f} reward={reward:.1f}")
            scores.append(reward)
        except TimeoutError:
            print("Timeout")
            scores.append(-1.0)
        except Exception as e:
            print(f"Exception = {str(e)}")
            openenv_client = None  # Force reconnect
            scores.append(-3.0)
    return scores

In [ ]:
# Quick test
test_good = [[{"content": json.dumps({
    "vertices_coords": [[0,0],[1,0],[1,1],[0,1]],
    "edges_vertices": [[0,1],[1,2],[2,3],[3,0],[0,2]],
    "edges_assignment": ["B","B","B","B","V"],
    "edges_foldAngle": [0,0,0,0,180]
})}]]
test_bad = [[{"content": "not json at all"}]]

print(f"valid_fold — good: {valid_fold(test_good)}, bad: {valid_fold(test_bad)}")
print(f"shape_match — good: {shape_match(test_good)}")
print(f"shape_match — bad: {shape_match(test_bad)}")

## 8. Train on Modal (recommended)

Run training on Modal instead of this Colab runtime — B200 GPU, Qwen3-32B BF16, no quantization needed.

The env server is already running on Railway so Modal just needs the URL.

In [ ]:
import subprocess, sys, os

# Install Modal if needed
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modal"], check=True)

# Authenticate (opens browser on first run — skip if already authed)
# subprocess.run(["modal", "setup"], check=True)

# Point Modal at the already-running Railway env server
RAILWAY_URL = "https://origami-env-production.up.railway.app"

result = subprocess.run(
    [
        "modal", "run", "modal_train.py",
        "--task", "all",
        "--max-steps", "600",
        "--server-url", RAILWAY_URL,
    ],
    cwd=os.path.dirname(os.path.abspath("modal_train.py")),  # repo root
)
print("Exit code:", result.returncode)

## 8. GRPO Trainer

In [ ]:
max_prompt_length = maximum_length + 1
max_completion_length = max_seq_length - max_prompt_length

from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    temperature = 1.0,
    learning_rate = 2e-4,
    weight_decay = 0.001,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,   # Smoother training with multi-task
    num_generations = 2,               # Decrease if out of memory
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    max_steps = 1000,                  # 4 tasks × 250 = more steps needed
    save_steps = 200,
    output_dir = "outputs",
    report_to = "none",
)

In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [
        valid_fold,
        shape_match,
    ],
    args = training_args,
    train_dataset = dataset,
)

## 9. Train!

In [ ]:
trainer.train()

## 10. Test After Training

In [ ]:
# Test all 4 tasks after training
from transformers import TextStreamer

for task_name, prompt_text in task_prompts.items():
    print(f"\n{'='*60}")
    print(f"Task: {task_name}")
    print(f"{'='*60}")
    text = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_text}],
        tokenize = False,
        add_generation_prompt = True,
    )
    _ = model.generate(
        **tokenizer(text, return_tensors="pt").to("cuda"),
        temperature = 1.0,
        max_new_tokens = 256,
        streamer = TextStreamer(tokenizer, skip_prompt = True),
    )

## 11. Save Model

In [ ]:
# Merge and save in mxfp4 4bit format
if False:
    model.save_pretrained_merged("finetuned_model", tokenizer, save_method = "mxfp4")
if False:
    model.push_to_hub_merged("repo_id/repo_name", tokenizer, token = "hf...", save_method = "mxfp4")

# Merge and save in 16bit
if False:
    model.save_pretrained_merged("finetuned_model", tokenizer, save_method = "merged_16bit")
if False:
    model.push_to_hub_merged("hf/origami-fold", tokenizer, save_method = "merged_16bit", token = "")